# Predictive Maintenance Using NASA C-MAPSS

## Phase 2 — Data Cleaning and Feature Engineering

**Medallion Architecture: Silver and Gold Layers**

Clean and validate the Bronze data, calculate Remaining Useful Life (RUL), and prepare engineered features for machine learning.

In [0]:
from pyspark.sql import functions as F

schema = "workspace.predictive_maintenance_rul"

# Load the Bronze tables created in Phase 1
train_bronze = spark.table(f"{schema}.bronze_train_raw")
test_bronze = spark.table(f"{schema}.bronze_test_raw")
test_rul_bronze = spark.table(f"{schema}.bronze_test_rul_raw")

# Display dataset dimensions
print("Training rows:", train_bronze.count())
print("Training columns:", len(train_bronze.columns))

print("Test rows:", test_bronze.count())
print("Test columns:", len(test_bronze.columns))

print("Official test RUL rows:", test_rul_bronze.count())

# Inspect sample records
display(train_bronze.limit(5))

Training rows: 20631
Training columns: 26
Test rows: 13096
Test columns: 26
Official test RUL rows: 100


unit_number,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.419
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.0,23.4236
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393.0,2388.0,100.0,38.9,23.4044


In [0]:
# 1. Check missing values in training data
missing_train = train_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in train_bronze.columns
])

print("Missing values in training data:")
display(missing_train)

# 2. Check missing values in test data
missing_test = test_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in test_bronze.columns
])

print("Missing values in test data:")
display(missing_test)

# 3. Check duplicate engine-cycle records
train_duplicates = (
    train_bronze.groupBy("unit_number", "cycle")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

test_duplicates = (
    test_bronze.groupBy("unit_number", "cycle")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate training engine-cycle groups:", train_duplicates)
print("Duplicate test engine-cycle groups:", test_duplicates)

# 4. Check invalid cycle numbers
print(
    "Training rows with cycle <= 0:",
    train_bronze.filter(F.col("cycle") <= 0).count()
)

print(
    "Test rows with cycle <= 0:",
    test_bronze.filter(F.col("cycle") <= 0).count()
)

Missing values in training data:


unit_number,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Missing values in test data:


unit_number,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Duplicate training engine-cycle groups: 0
Duplicate test engine-cycle groups: 0
Training rows with cycle <= 0: 0
Test rows with cycle <= 0: 0


In [0]:
# Clean training and test data
train_silver = (
    train_bronze
    .dropna(subset=train_bronze.columns)
    .filter((F.col("unit_number") > 0) & (F.col("cycle") > 0))
)

test_silver = (
    test_bronze
    .dropna(subset=test_bronze.columns)
    .filter((F.col("unit_number") > 0) & (F.col("cycle") > 0))
)

# Verify data quality
print("Training rows before cleaning:", train_bronze.count())
print("Training rows after cleaning:", train_silver.count())

print("Test rows before cleaning:", test_bronze.count())
print("Test rows after cleaning:", test_silver.count())

# Save Silver Delta tables
train_silver.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.silver_train_clean"
)

test_silver.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.silver_test_clean"
)

# Save official test RUL labels as a Silver table
test_rul_silver = test_rul_bronze.dropna(
    subset=["official_rul"]
)

test_rul_silver.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.silver_test_rul"
)

print("Silver tables created successfully.")
print("Official RUL label rows:", test_rul_silver.count())

Training rows before cleaning: 20631
Training rows after cleaning: 20631
Test rows before cleaning: 13096
Test rows after cleaning: 13096
Silver tables created successfully.
Official RUL label rows: 100


In [0]:
# Find the last observed cycle for each training engine
train_max_cycles = (
    train_silver
    .groupBy("unit_number")
    .agg(F.max("cycle").alias("max_cycle"))
)

# Calculate RUL for each training record
train_with_rul = (
    train_silver
    .join(train_max_cycles, on="unit_number", how="inner")
    .withColumn(
        "RUL",
        (F.col("max_cycle") - F.col("cycle")).cast("double")
    )
)

# Check the RUL calculation
print("Training rows with RUL:", train_with_rul.count())
print("Minimum RUL:", train_with_rul.agg(F.min("RUL")).first()[0])
print("Maximum RUL:", train_with_rul.agg(F.max("RUL")).first()[0])

display(
    train_with_rul
    .filter(F.col("unit_number") == 1)
    .orderBy("cycle")
    .select("unit_number", "cycle", "max_cycle", "RUL")
    .limit(5)
)

Training rows with RUL: 20631
Minimum RUL: 0.0
Maximum RUL: 361.0


unit_number,cycle,max_cycle,RUL
1,1,192,191.0
1,2,192,190.0
1,3,192,189.0
1,4,192,188.0
1,5,192,187.0


In [0]:
train_with_rul.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.silver_train_labeled"
)

print(
    "Saved Silver training rows:",
    spark.table(f"{schema}.silver_train_labeled").count()
)

display(
    spark.table(f"{schema}.silver_train_labeled")
    .filter(F.col("unit_number") == 1)
    .orderBy("cycle")
    .select("unit_number", "cycle", "max_cycle", "RUL")
    .limit(5)
)

Saved Silver training rows: 20631


unit_number,cycle,max_cycle,RUL
1,1,192,191.0
1,2,192,190.0
1,3,192,189.0
1,4,192,188.0
1,5,192,187.0


In [0]:
from pyspark.sql.window import Window

# Build features using each engine's cycle history
def add_features(df):
    window_3 = (
        Window.partitionBy("unit_number")
        .orderBy("cycle")
        .rowsBetween(-2, 0)
    )

    result = df

    for i in range(1, 22):
        sensor = f"sensor_{i}"

        result = (
            result
            .withColumn(
                f"{sensor}_rolling_avg_3",
                F.avg(F.col(sensor)).over(window_3)
            )
            .withColumn(
                f"{sensor}_change",
                F.col(sensor) - F.lag(F.col(sensor), 1).over(
                    Window.partitionBy("unit_number").orderBy("cycle")
                )
            )
        )

    return result

# Gold training features include the RUL target
gold_train = add_features(train_with_rul)

# Gold test features use only observed test data
gold_test = add_features(test_silver)

print("Gold training rows:", gold_train.count())
print("Gold training columns:", len(gold_train.columns))
print("Gold test rows:", gold_test.count())
print("Gold test columns:", len(gold_test.columns))

Gold training rows: 20631
Gold training columns: 70
Gold test rows: 13096
Gold test columns: 68


In [0]:
# Save Gold feature tables
gold_train.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.gold_train_features"
)

gold_test.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.gold_test_features"
)

print(
    "Gold training table rows:",
    spark.table(f"{schema}.gold_train_features").count()
)

print(
    "Gold test table rows:",
    spark.table(f"{schema}.gold_test_features").count()
)

print("Gold feature tables saved successfully.")

Gold training table rows: 20631
Gold test table rows: 13096
Gold feature tables saved successfully.


In [0]:
# Inspect the final Gold training features
display(
    spark.table(f"{schema}.gold_train_features")
    .filter(F.col("unit_number") == 1)
    .orderBy("cycle")
    .select(
        "unit_number",
        "cycle",
        "sensor_2",
        "sensor_2_rolling_avg_3",
        "sensor_2_change",
        "RUL"
    )
    .limit(5)
)

unit_number,cycle,sensor_2,sensor_2_rolling_avg_3,sensor_2_change,RUL
1,1,641.82,641.82,null,191.0
1,2,642.15,641.985,0.32999999999992724,190.0
1,3,642.35,642.1066666666667,0.20000000000004547,189.0
1,4,642.35,642.2833333333333,0.0,188.0
1,5,642.37,642.3566666666667,0.01999999999998181,187.0
